# Calendário de Espectrogramas de gravações no Google Drive

Este notebook executa o [**spectro-calendar**](https://github.com/biodiversica/spectro-calendar) sobre áudios armazenados no seu **Google Drive** e gera um **calendário de espectrogramas em HTML** — uma miniatura por gravação, com as datas nas colunas e os horários nas linhas, permitindo visualizar semanas de monitoramento de uma só vez.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para acessar as gravações
2. **Instala o spectro-calendar e o ffmpeg** automaticamente
3. **Faz uma busca na pasta de áudios** e mostra exatamente quais gravações serão incluídas
4. **Gera os espectrogramas e o calendário em HTML**
5. **Abre o calendário dentro do notebook**, com os players de áudio funcionando
6. **Empacota tudo em um `.zip`** para salvar no seu Drive e/ou baixar

### Estrutura de pastas:
```
# Todas as gravações em uma pasta
MyDrive/audio/
  20250101_180000.WAV
  20250101_183000.WAV

# Subpastas — uma pasta por gravador (ative "incluir subpastas" no Passo 2)
MyDrive/audio/
  PONTO_A/
    20250101_180000.WAV
  PONTO_B/
    20250101_180000.WAV
```

### Formato dos nomes de arquivo:
Cada arquivo `.wav` precisa ter a data e a hora da gravação no nome:
- **AudioMoth** (padrão): `YYYYMMDD_HHMMSS.WAV` — ex.: `20250615_203000.WAV`
- **Com prefixo**: `SM4_20250615_203000.wav` — configure o prefixo como `SM4_`
- **Qualquer outro formato**: configure o formato de data/hora, ex.: `%Y-%m-%d_%H-%M-%S` para `2025-06-15_20-30-00.wav`

Todo o calendário é montado a partir desses timestamps, então **um nome de arquivo que não pode ser lido interrompe a execução** — o notebook indica quais são os arquivos no Passo 3.

### Sobre o calendário e o áudio:
- O Google Drive **não exibe páginas HTML**: abrir o `index_.html` por lá apenas faz o download do arquivo. Por isso o calendário final é entregue como um **`.zip`** — descompacte no seu computador e abra o `index_.html` em um navegador.
- Os **players de áudio funcionam dentro deste notebook** (Passo 5): um pequeno servidor web rodando no ambiente do Colab entrega tanto a página quanto as gravações diretamente do seu Drive, sem copiar nada.
- Para que os players continuem funcionando **depois do download**, o Passo 6 pode copiar as gravações para dentro do zip. Isso torna o calendário autossuficiente, mas bem maior — WAV não é comprimido, e um minuto de AudioMoth a 48 kHz tem cerca de 23 MB.

### Antes de começar:
- Uma **conta Google**, com as gravações no Google Drive
- Apenas arquivos `.wav` / `.WAV` são lidos; outros formatos são ignorados

---
## Passo 1 — Conectar ao Google Drive e instalar as libs

Execute as duas células abaixo. A primeira vai pedir para **permitir o acesso ao seu Google Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive conectado com sucesso.")

In [ ]:
#@title 📦 Instalar o spectro-calendar { display-mode: "form" }

#@markdown Instala o pacote [spectro-calendar](https://github.com/biodiversica/spectro-calendar)
#@markdown e verifica se o `ffmpeg` está disponível (o Colab normalmente já o traz instalado).
#@markdown Execute esta célula novamente após reiniciar o ambiente.

import shutil
import subprocess
import sys

!pip install -q "git+https://github.com/biodiversica/spectro-calendar.git"

if shutil.which("ffmpeg") is None:
    print("ffmpeg não encontrado, instalando (leva cerca de um minuto) ...")
    !apt-get -qq update
    !apt-get -qq install -y ffmpeg

_ffmpeg = shutil.which("ffmpeg")
if _ffmpeg:
    _version = subprocess.run([_ffmpeg, "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(f"ffmpeg : {_version}")
else:
    print("ffmpeg NÃO está disponível -- o backend scipy, mais lento, será usado no lugar.")

_check = subprocess.run([sys.executable, "-m", "spectro_calendar", "--help"], capture_output=True, text=True)
if _check.returncode == 0:
    import spectro_calendar
    print(f"spectro-calendar {spectro_calendar.__version__} instalado com sucesso.")
else:
    print("O spectro-calendar não foi instalado corretamente. O erro foi:")
    print(_check.stderr[-2000:])

---
## Passo 2 — Configuração

Preencha os quatro formulários abaixo. **Não é necessário editar nenhum código** — basta digitar ou selecionar os valores e executar cada célula, de cima para baixo:

1. **Entrada de áudio** — onde estão as gravações e como os nomes dos arquivos são formatados
2. **Filtros de data e hora** — quais gravações entram no calendário
3. **Configurações do espectrograma** — a aparência de cada miniatura
4. **Saída, áudio e anotação** — o que a execução produz

> **Dica:** os formulários escondem o código automaticamente. Para ver o código, clique no ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [ ]:
#@title 🗂️ Entrada de áudio { display-mode: "form" }

from datetime import datetime
from pathlib import Path

#@markdown **Pasta de áudios** — a pasta do seu Google Drive com as gravações.
#@markdown Exemplo: `/content/drive/MyDrive/meu_projeto/audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown **Incluir subpastas** — também lê os arquivos `.wav` dentro de subpastas, ex.: uma pasta
#@markdown por gravador. As imagens de cada gravação vão para a subpasta correspondente na saída.
RECURSIVE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Prefixo do nome do arquivo** — o texto que vem antes da data/hora no nome do arquivo,
#@markdown **incluindo** o separador. Deixe em branco para nomes padrão do AudioMoth.
#@markdown Exemplo: `SM4_` para `SM4_20250615_203000.wav`
FILENAME_PREFIX = ""  #@param {type:"string"}

#@markdown **Formato de data/hora** — como o timestamp está escrito, depois de remover o prefixo
#@markdown ([códigos strptime](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes)).
#@markdown `%Y%m%d_%H%M%S` corresponde a `20250615_203000`, e `%Y-%m-%d_%H-%M-%S` a `2025-06-15_20-30-00`.
DATETIME_FORMAT = "%Y%m%d_%H%M%S"  #@param {type:"string"}

REC_DIR = Path(DRIVE_INPUT_DIR.strip().rstrip("/"))
if not REC_DIR.is_dir():
    raise FileNotFoundError(f"Pasta de áudios não encontrada (confira o caminho neste formulário): {REC_DIR}")

_example = FILENAME_PREFIX + datetime(2025, 6, 15, 20, 30, 0).strftime(DATETIME_FORMAT) + ".WAV"

print(f"Pasta de áudios : {REC_DIR}")
print(f"Subpastas       : {RECURSIVE}")
print(f"Nome esperado   : {_example}")

In [ ]:
#@title 📅 Filtros de data e hora { display-mode: "form" }

#@markdown Todos os filtros são opcionais — com os valores padrão, **todas** as gravações entram no
#@markdown calendário. Estes filtros são a principal forma de reduzir o tempo de execução: cada
#@markdown arquivo selecionado é lido por inteiro do seu Drive.

#@markdown ---
#@markdown **Intervalo de datas** — `YYYYMMDD`. Deixe um dos campos em branco para não limitar aquele lado.
START_DATE = ""  #@param {type:"string"}
END_DATE = ""  #@param {type:"string"}

#@markdown **Datas específicas** (opcional) — uma lista `YYYYMMDD` separada por vírgulas, ex.: `20250615, 20250617`.
#@markdown Quando preenchido, **substitui** o intervalo de datas acima (os dois não podem ser combinados).
DATES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Janela de horários** — `HHMMSS`. Apenas as gravações feitas entre esses dois horários
#@markdown são incluídas, em todas as datas selecionadas.
START_TIME = "000000"  #@param {type:"string"}
END_TIME = "235900"  #@param {type:"string"}

#@markdown **Passo de tempo (minutos)** — mantém apenas uma gravação por intervalo em cada dia;
#@markdown `0` mantém todas. Ex.: `30` transforma uma gravação a cada 5 minutos em uma a cada 30.
TIME_STEP = 0  #@param {type:"integer"}

START_DATE = START_DATE.strip()
END_DATE = END_DATE.strip()
START_TIME = START_TIME.strip() or "000000"
END_TIME = END_TIME.strip() or "235900"
DATES_LIST = [d.strip() for d in DATES.replace(";", ",").split(",") if d.strip()]
TIME_STEP = max(0, int(TIME_STEP))

if DATES_LIST and (START_DATE or END_DATE):
    raise ValueError("Use 'datas específicas' OU um intervalo de datas, não os dois.")
for _value in DATES_LIST + [d for d in (START_DATE, END_DATE) if d]:
    if not (_value.isdigit() and len(_value) == 8):
        raise ValueError(f"As datas devem ser escritas como YYYYMMDD; valor recebido: {_value}")
for _value in (START_TIME, END_TIME):
    if not (_value.isdigit() and len(_value) == 6):
        raise ValueError(f"Os horários devem ser escritos como HHMMSS; valor recebido: {_value}")

if DATES_LIST:
    print(f"Datas              : {', '.join(DATES_LIST)}")
else:
    print(f"Intervalo de datas : {START_DATE or '...'} -> {END_DATE or '...'}")
print(f"Janela de horários : {START_TIME} -> {END_TIME}")
print(f"Passo de tempo     : {TIME_STEP or '-'}")

In [ ]:
#@title 🎨 Configurações do espectrograma { display-mode: "form" }

#@markdown **Usar ffmpeg** — o backend rápido de espectrogramas, e o único que respeita as opções de
#@markdown cor, ganho e escala de frequência abaixo. Se o ffmpeg não estiver disponível, a ferramenta
#@markdown usa o scipy automaticamente (mais lento, sempre com a paleta `plasma` em escala linear).
USE_FFMPEG = True  #@param {type:"boolean"}

#@markdown **Processos em paralelo** — quantas gravações são processadas ao mesmo tempo (`1` desativa o
#@markdown paralelismo). Um ambiente do Colab normalmente tem 2 CPUs; mais processos ainda ajudam um
#@markdown pouco, já que cada um passa a maior parte do tempo esperando o Drive.
MAX_CORES = 4  #@param {type:"integer"}

#@markdown ---
#@markdown **Faixa de frequência (Hz)** — o eixo vertical de cada espectrograma. Restringi-la à faixa
#@markdown dos sons de interesse deixa as miniaturas muito mais legíveis.
LOWEST_FREQ = 0  #@param {type:"integer"}
HIGHEST_FREQ = 20000  #@param {type:"integer"}

#@markdown **Escala de frequência** — `log` dá mais espaço às frequências baixas (somente ffmpeg).
FREQ_SCALE = "lin"  #@param ["lin", "log"]

#@markdown ---
#@markdown **Ganho (dB)** — clareia o espectrograma; aumente para gravações baixas (somente ffmpeg).
GAIN = 1  #@param {type:"integer"}

#@markdown **Escala do ganho** — como as magnitudes são mapeadas em cores (somente ffmpeg).
GAIN_SCALE = "log"  #@param ["log", "sqrt", "cbrt", "4thrt", "5thrt", "lin"]

#@markdown **Paleta de cores** — as cores do espectrograma (somente ffmpeg).
COLOR_CHOICE = "plasma"  #@param ["plasma", "viridis", "magma", "cividis", "intensity", "rainbow", "moreland", "nebulae", "fire", "fiery", "fruit", "cool", "green", "terrain", "channel"]

#@markdown ---
#@markdown **Imagem em tamanho real** (`largura x altura`) e **miniatura** (`largura:altura`), em pixels.
#@markdown A miniatura é reduzida a partir da imagem em tamanho real, então o primeiro valor também
#@markdown influencia o nível de detalhe da miniatura. O tamanho da miniatura é o tamanho da célula do
#@markdown calendário — células maiores deixam a página mais clara, porém mais pesada.
IMG_SIZE = "1080x720"  #@param {type:"string"}
THUMBNAIL_SCALE = "108:72"  #@param {type:"string"}

#@markdown **Guardar as imagens em tamanho real** — salva também cada espectrograma em resolução total
#@markdown junto da miniatura, para abrir quando uma célula precisar de um olhar mais atento.
#@markdown Desativado por padrão: o calendário usa apenas as miniaturas, e os PNGs em tamanho real
#@markdown ocupam muito mais espaço no zip.
SAVE_FULLSIZE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Rótulo do espectrograma** (opcional) — um sufixo que separa várias execuções dentro do mesmo
#@markdown calendário, ex.: `lf` e `hf` para uma faixa de frequências baixas e outra de altas. Ele nomeia
#@markdown a página (`index_lf.html`) e todas as imagens que ela contém.
SPEC_LABEL = ""  #@param {type:"string"}

MAX_CORES = max(1, int(MAX_CORES))
SPEC_LABEL = SPEC_LABEL.strip()

print(f"Backend             : {'ffmpeg' if USE_FFMPEG else 'scipy'}")
print(f"Faixa de frequência : {LOWEST_FREQ} - {HIGHEST_FREQ} Hz ({FREQ_SCALE})")
print(f"Miniatura           : {THUMBNAIL_SCALE.replace(':', ' x ')} px")
print(f"Processos           : {MAX_CORES}")
print(f"Rótulo              : {SPEC_LABEL or '-'}")

In [ ]:
#@title 📦 Saída, áudio e anotação { display-mode: "form" }

from pathlib import Path

#@markdown **Nome do calendário** — nomeia a pasta com as imagens geradas, a página HTML e a folha de
#@markdown estilo, além do zip criado a partir dela no Passo 6.
CALENDAR_NAME = "my_calendar"  #@param {type:"string"}

#@markdown ---
#@markdown **Players de áudio** — adiciona um player abaixo de cada miniatura. Eles reproduzem o áudio
#@markdown direto do seu Drive na visualização do notebook (Passo 5); para que continuem funcionando
#@markdown depois do download, inclua o áudio no zip no Passo 6.
INCLUDE_AUDIO = True  #@param {type:"boolean"}

#@markdown **Modo de anotação** — adiciona uma caixa de seleção abaixo de cada miniatura e uma barra de
#@markdown ferramentas que exporta as gravações marcadas em uma lista `.txt`. Use `Shift+clique` em uma
#@markdown segunda caixa para selecionar todo o intervalo entre elas.
ANNOTATE = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Começar do zero** — apaga as imagens de uma execução anterior deste mesmo calendário antes
#@markdown de gerar. Deixe desmarcado para retomar uma execução interrompida: com o backend ffmpeg,
#@markdown as gravações que já têm miniatura são puladas.
CLEAR = False  #@param {type:"boolean"}

#@markdown ---
#@markdown Tudo é gerado no disco local do ambiente, que é rápido, porém temporário.
#@markdown É o Passo 6 que leva o resultado para o seu Drive.

CALENDAR_NAME = (CALENDAR_NAME.strip() or "my_calendar").replace(" ", "_").replace("/", "-")
OUT_DIR = Path("/content/spectro_calendar") / CALENDAR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pasta de trabalho : {OUT_DIR}")
print(f"Players de áudio  : {INCLUDE_AUDIO}")
print(f"Modo de anotação  : {ANNOTATE}")

---
## Passo 3 — Buscar as gravações

Esta célula lista os arquivos `.wav` da sua pasta de áudios, lê o timestamp de cada nome de arquivo e aplica os filtros do Passo 2, para que você veja **exatamente o que o calendário vai conter antes de gerar qualquer coisa**.

Ela também informa quanto de áudio precisa ser lido do Drive. Essa leitura é a parte lenta de todo o processo — se o número parecer alto, volte ao Passo 2 e reduza o intervalo de datas, a janela de horários ou o passo de tempo.

In [ ]:
#@title 🔍 Busca { display-mode: "form" }

from spectro_calendar.cli import (
    filter_dates_by_range,
    filter_wav_files_by_time_step,
    filter_wav_files_by_time_window,
    get_available_dates,
    parse_recording_datetime,
    parse_time_string,
)

_candidates = REC_DIR.rglob("*") if RECURSIVE else REC_DIR.iterdir()
ALL_WAVS = [p for p in _candidates if p.is_file() and p.suffix.lower() == ".wav"]
if not ALL_WAVS:
    raise FileNotFoundError("Nenhum arquivo .wav encontrado na pasta de áudios. Se as gravações estão em subpastas, ative 'incluir subpastas' no Passo 2.")

FILE_DATES = {}
_unreadable = []
for _wav in ALL_WAVS:
    try:
        FILE_DATES[_wav] = parse_recording_datetime(_wav, DATETIME_FORMAT, FILENAME_PREFIX)
    except ValueError:
        _unreadable.append(_wav)

if _unreadable:
    print("Não foi possível ler a data e a hora destes nomes de arquivo:")
    for _wav in _unreadable[:10]:
        print(f"   {_wav.name}")
    if len(_unreadable) > 10:
        print(f"   ... (+{len(_unreadable) - 10})")
    raise ValueError("Ajuste o prefixo e/ou o formato de data/hora no Passo 2, ou mova estes arquivos para fora da pasta.")

ALL_WAVS.sort(key=lambda w: FILE_DATES[w])
_available = get_available_dates(FILE_DATES)

if DATES_LIST:
    _missing = [d for d in DATES_LIST if d not in _available]
    if _missing:
        raise ValueError(f"Nenhuma gravação nas datas solicitadas: {', '.join(_missing)}")
    _dates = DATES_LIST
elif START_DATE or END_DATE:
    _dates = filter_dates_by_range(_available, START_DATE or None, END_DATE or None)
else:
    _dates = _available

SELECTED_WAVS = filter_wav_files_by_time_window(
    ALL_WAVS,
    FILE_DATES,
    parse_time_string(START_TIME).time(),
    parse_time_string(END_TIME).time(),
    _dates,
)
if TIME_STEP:
    SELECTED_WAVS = filter_wav_files_by_time_step(SELECTED_WAVS, FILE_DATES, _dates, TIME_STEP)
if not SELECTED_WAVS:
    raise ValueError("Nenhuma gravação corresponde aos filtros do Passo 2. Amplie o intervalo de datas, a janela de horários ou o passo de tempo.")

# One calendar cell per date and time of day, exactly as the table is built.
_cells = {(FILE_DATES[w].strftime("%Y%m%d"), FILE_DATES[w].strftime("%H%M%S")) for w in SELECTED_WAVS}
_columns = sorted({cell[0] for cell in _cells})
_rows = sorted({cell[1] for cell in _cells})
_bytes = sum(w.stat().st_size for w in SELECTED_WAVS)

print(f"Gravações encontradas : {len(ALL_WAVS)}")
print(f"Selecionadas          : {len(SELECTED_WAVS)}")
print(f"Datas                 : {_columns[0]} -> {_columns[-1]}  ({len(_columns)} dia(s))")
print(f"Horários do dia       : {_rows[0][:2]}:{_rows[0][2:4]} -> {_rows[-1][:2]}:{_rows[-1][2:4]}  ({len(_rows)})")
print(f"Tabela do calendário  : {len(_columns)} coluna(s) x {len(_rows)} linha(s)")
print(f"Áudio a ser lido      : {_bytes / 2 ** 30:.2f} GB")

if len(SELECTED_WAVS) > len(_cells):
    print(f"\n{len(SELECTED_WAVS) - len(_cells)} gravação(ões) compartilham data e hora com outra; o calendário tem uma única célula por horário, então apenas uma de cada par aparece na tabela (todas as miniaturas continuam sendo geradas).")

---
## Passo 4 — Gerar o calendário

Este é o passo principal: cada gravação selecionada é lida do Drive, transformada em uma miniatura de espectrograma e posicionada no calendário HTML.

Tudo é gravado no **disco local do ambiente** (`/content/spectro_calendar/...`), e não diretamente no seu Drive — escrever milhares de PNGs pequenos no Drive é muito mais lento. O Passo 6 empacota o resultado em um único arquivo zip, que é o que vai para o seu Drive.

> Dependendo da quantidade de gravações selecionadas, este passo pode demorar. O progresso é exibido abaixo.
>
> Se a execução for interrompida (ou a sessão do Colab cair), basta executar novamente com **começar do zero** desmarcado: com o backend ffmpeg, as gravações que já têm miniatura são puladas, e a execução continua de onde parou.

In [ ]:
#@title 🚀 Gerar o calendário { display-mode: "form" }

#@markdown **Mostrar cada arquivo** — imprime uma linha por gravação em vez de uma contagem.
#@markdown Útil para acompanhar execuções pequenas; muito verboso em execuções grandes.
SHOW_EVERY_FILE = False  #@param {type:"boolean"}

import os
import re
import shlex
import subprocess
import sys
import time

cmd = [sys.executable, "-m", "spectro_calendar", str(REC_DIR), "--output-dir", str(OUT_DIR)]
if RECURSIVE:
    cmd.append("--recursive")
if FILENAME_PREFIX:
    cmd += ["--filename-prefix", FILENAME_PREFIX]
cmd += ["--datetime-format", DATETIME_FORMAT]
if DATES_LIST:
    cmd += ["--dates", *DATES_LIST]
else:
    if START_DATE:
        cmd += ["--start-date", START_DATE]
    if END_DATE:
        cmd += ["--end-date", END_DATE]
cmd += ["--start-time", START_TIME, "--end-time", END_TIME]
if TIME_STEP:
    cmd += ["--time-step", str(TIME_STEP)]
if USE_FFMPEG:
    cmd.append("--use-ffmpeg")
cmd += [
    "--max-cores", str(MAX_CORES),
    "--lowest-freq", str(LOWEST_FREQ),
    "--highest-freq", str(HIGHEST_FREQ),
    "--freq-scale", FREQ_SCALE,
    "--gain", str(GAIN),
    "--gain-scale", GAIN_SCALE,
    "--color-choice", COLOR_CHOICE,
    "--img-size", IMG_SIZE,
    "--thumbnail-scale", THUMBNAIL_SCALE,
]
if SPEC_LABEL:
    cmd += ["--spec-label", SPEC_LABEL]
if SAVE_FULLSIZE:
    cmd.append("--save-fullsize")
if CLEAR:
    cmd.append("--clear")
if INCLUDE_AUDIO:
    cmd.append("--include-audio")
if ANNOTATE:
    cmd.append("--annotate")

print(f"Executando:\n  spectro-calendar {shlex.join(cmd[3:])}\n")

# One line per file is far too much output for a month of recordings; count
# those lines instead and keep everything else (warnings, errors) visible.
_per_file = re.compile(r"^(Processing |making (thumbnail|fullsize) spectrogram for|Spectrogram for )")
_finished = re.compile(r"^(making thumbnail spectrogram for|Spectrogram for )")
_done = 0
_start = time.time()
_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
_counting = False  # the running count is on screen, with no line break yet
for _line in _process.stdout:
    if SHOW_EVERY_FILE or not _per_file.match(_line):
        # A warning (e.g. an unreadable recording) goes on a line of its own
        print(("\n" if _counting else "") + _line, end="")
        _counting = False
    elif _finished.match(_line):
        _done += 1
        print(f"\r   {_done}/{len(SELECTED_WAVS)} espectrogramas prontos   ", end="", flush=True)
        _counting = True
_process.wait()
if _counting:
    print()

if _process.returncode != 0:
    raise RuntimeError(f"O spectro-calendar falhou -- veja a mensagem acima (exit code {_process.returncode})")

INDEX_PAGES = sorted(OUT_DIR.glob("index_*.html"))
print(f"Tempo total          : {time.time() - _start:.0f} s")
print(f"Imagens geradas      : {len(list(OUT_DIR.rglob('*.png')))}")
for _page in INDEX_PAGES:
    print(f"Página do calendário : {_page}")

---
## Passo 5 — Abrir o calendário dentro do notebook

A célula abaixo inicia um pequeno servidor web dentro do ambiente do Colab e exibe o calendário aqui mesmo, em um quadro. O botão **Tela cheia** acima dele dá a tela inteira a um calendário grande (o Colab não consegue abri-lo em uma aba separada do navegador: o endereço da visualização só funciona dentro do notebook).

Este é o único lugar em que **os players de áudio funcionam sem copiar nada**: cada gravação é lida do seu Drive, sob demanda, apenas quando você aperta o play (uma gravação longa leva alguns instantes para carregar). O mesmo vale para o modo de anotação — marcar as caixas, selecionar intervalos com `Shift+clique` e exportar o `.txt` funcionam aqui, e a seleção é lembrada entre recarregamentos da página.

A visualização só funciona enquanto **este ambiente estiver ativo** e apenas para a sua conta Google. Quando a sessão terminar, use o zip do Passo 6.

In [ ]:
#@title 👀 Abrir o calendário { display-mode: "form" }

#@markdown Inicia um servidor web dentro do ambiente do Colab e exibe o calendário abaixo.
#@markdown Os players de áudio reproduzem direto do seu Drive — nada é copiado.

#@markdown **Altura do quadro** (pixels) — o tamanho da visualização abaixo.
PREVIEW_HEIGHT = 700  #@param {type:"integer"}

#@markdown **Porta** — só vale a pena mudar se esta já estiver ocupada no ambiente.
PREVIEW_PORT = 8765  #@param {type:"integer"}

import functools
import os
import re
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer

from IPython.display import HTML, display


AUDIO_SCRIPT = """\
// Colab's proxy passes the page's own requests (the thumbnails load fine) but
// not the browser's media requests, so an <audio> pointed straight at a WAV
// never starts. Fetch the recording as an ordinary request and play it from
// memory instead, keeping only the last one played in memory.
(function () {
  var loaded = null;

  function note(audio, text) {
    var label = audio.nextElementSibling;
    if (!label || !label.classList.contains("audio-status")) {
      label = document.createElement("div");
      label.className = "audio-status";
      label.style.cssText = "font-size:11px;color:#666";
      audio.insertAdjacentElement("afterend", label);
    }
    label.textContent = text;
  }

  function release(audio) {
    audio.pause();
    URL.revokeObjectURL(audio.dataset.blob);
    delete audio.dataset.blob;
    audio.removeAttribute("src");
    audio.load();
  }

  document.querySelectorAll("audio").forEach(function (audio) {
    var source = audio.querySelector("source");
    if (!source) return;
    var url = source.src;
    audio.addEventListener("play", function () {
      if (audio.dataset.blob) return;
      audio.pause();
      if (audio.dataset.busy) return;
      audio.dataset.busy = "1";
      note(audio, "carregando...");
      fetch(url).then(function (response) {
        if (!response.ok) throw new Error("HTTP " + response.status);
        return response.blob();
      }).then(function (blob) {
        if (loaded && loaded !== audio) release(loaded);
        loaded = audio;
        audio.dataset.blob = URL.createObjectURL(new Blob([blob], {type: "audio/wav"}));
        audio.src = audio.dataset.blob;
        note(audio, "");
        return audio.play().catch(function () { note(audio, "pronto, aperte o play"); });
      }).catch(function (error) {
        note(audio, "não foi possível carregar a gravação (" + error.message + ")");
      }).finally(function () {
        delete audio.dataset.busy;
      });
    });
  });
})();
"""


class CalendarHandler(SimpleHTTPRequestHandler):
    """Servidor de arquivos estáticos que entende requisições Range, para o navegador
    poder avançar dentro de um WAV sem precisar baixá-lo inteiro antes de tocar."""

    def log_message(self, *args):
        pass  # mantém a saída do notebook limpa

    def end_headers(self):
        self.send_header("Accept-Ranges", "bytes")
        super().end_headers()

    def send_calendar_page(self, path):
        """Serves a calendar page with AUDIO_SCRIPT added. Only the served copy
        changes: the file on disk, and so the zip from Step 6, stays as it is."""
        with open(path, encoding="utf-8") as source:
            page = source.read()
        if INCLUDE_AUDIO:
            page = page.replace("</body>", f"<script>\n{AUDIO_SCRIPT}</script>\n</body>", 1)
        body = page.encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path = self.translate_path(self.path)
        if os.path.basename(path).startswith("index_") and path.endswith(".html") and os.path.isfile(path):
            return self.send_calendar_page(path)

        _range = self.headers.get("Range", "")
        wanted = re.fullmatch(r"bytes=(\d*)-(\d*)", _range.strip()) if _range else None
        if not wanted or not os.path.isfile(path):
            return super().do_GET()

        size = os.path.getsize(path)
        first, last = wanted.group(1), wanted.group(2)
        if first == "":
            start, end = max(0, size - int(last or 0)), size - 1
        else:
            start, end = int(first), int(last) if last else size - 1
        end = min(end, size - 1)
        if start > end or start >= size:
            self.send_response(416)
            self.send_header("Content-Range", f"bytes */{size}")
            self.end_headers()
            return

        self.send_response(206)
        self.send_header("Content-Type", self.guess_type(path))
        self.send_header("Content-Range", f"bytes {start}-{end}/{size}")
        self.send_header("Content-Length", str(end - start + 1))
        self.end_headers()
        try:
            with open(path, "rb") as source:
                source.seek(start)
                remaining = end - start + 1
                while remaining > 0:
                    chunk = source.read(min(256 * 1024, remaining))
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    remaining -= len(chunk)
        except (BrokenPipeError, ConnectionResetError):
            pass  # o navegador encerrou a conexão (player fechado, página recarregada)


INDEX_PAGES = sorted(OUT_DIR.glob("index_*.html"))
if not INDEX_PAGES:
    raise FileNotFoundError("Nenhuma página de calendário encontrada -- execute o Passo 4 primeiro.")

# The page links to the recordings by a relative path that climbs out of the
# calendar folder, so the server has to be rooted at the first directory
# holding both the calendar and the audio -- usually /content.
SERVE_ROOT = os.path.commonpath([str(OUT_DIR.resolve()), str(REC_DIR.resolve())])

_running = globals().get("PREVIEW_SERVER")
if _running is not None:
    _running.shutdown()
    _running.server_close()

PREVIEW_SERVER = ThreadingHTTPServer(("", PREVIEW_PORT), functools.partial(CalendarHandler, directory=SERVE_ROOT))
PREVIEW_SERVER.daemon_threads = True
threading.Thread(target=PREVIEW_SERVER.serve_forever, daemon=True).start()

try:
    from google.colab.output import eval_js

    _base = eval_js(f"google.colab.kernel.proxyPort({PREVIEW_PORT})").rstrip("/")
    _urls = [f"{_base}/{os.path.relpath(page, SERVE_ROOT)}" for page in INDEX_PAGES]
    # Colab's preview address only works inside this output frame (browser
    # storage partitioning), so a link to a new tab would just give a 404.
    # Full screen stands in for it; where the browser refuses full screen
    # inside Colab, the frame grows to the height of the screen instead.
    _buttons = [
        '<button onclick="var f=document.getElementById(\'calendar-frame\');'
        ' (f.requestFullscreen ? f.requestFullscreen() : Promise.reject())'
        '.catch(function () { f.height = screen.availHeight - 120; });">⛶ Tela cheia</button>'
    ]
    if len(INDEX_PAGES) > 1:
        _buttons += [
            f'<button onclick="document.getElementById(\'calendar-frame\').src=\'{url}\'">{page.name}</button>'
            for url, page in zip(_urls, INDEX_PAGES)
        ]
    display(HTML(
        f'<p>{" ".join(_buttons)}</p>'
        f'<iframe id="calendar-frame" src="{_urls[0]}" width="100%" height="{PREVIEW_HEIGHT}"'
        ' allow="fullscreen; autoplay" allowfullscreen'
        ' style="border:1px solid #bbb;border-radius:4px"></iframe>'
    ))
except Exception as error:
    print(f"Não foi possível abrir a visualização aqui (esta célula só funciona no Google Colab). O calendário está no disco do ambiente, em:\n  {INDEX_PAGES[0]}\n  ({error})")

---
## Passo 6 — Empacotar em zip, salvar e baixar

A célula abaixo empacota o calendário (miniaturas, `index_*.html` e a folha de estilo) em um único `.zip`, copia para o seu Google Drive e/ou baixa para o seu computador.

Descompacte localmente e abra o arquivo `index_*.html` em um navegador.

**O que acontece com os players de áudio no zip?**
- **Desmarcado (padrão):** os players são removidos da página do zip — eles apontariam para o seu Drive e apareceriam como controles quebrados. O calendário em si, com todas as miniaturas, funciona normalmente offline.
- **Marcado:** cada gravação usada no calendário é copiada para dentro do zip, na pasta `audio/`, e os links são reescritos para apontar para lá. O calendário fica completamente autossuficiente, mas o zip cresce o tamanho de todo esse áudio — confira o valor de "áudio a ser lido" do Passo 3 antes de ativar.

In [ ]:
#@title 📦 Empacotar, salvar e baixar { display-mode: "form" }

#@markdown **Incluir o áudio no zip** — copia para dentro do zip todas as gravações usadas no calendário,
#@markdown para que os players continuem funcionando offline. Isso soma ao zip o tamanho total desse
#@markdown áudio (veja "áudio a ser lido" no Passo 3). Com esta opção desmarcada, os players são
#@markdown removidos da página do zip, já que apontariam apenas de volta para o seu Drive.
BUNDLE_AUDIO_IN_ZIP = False  #@param {type:"boolean"}

#@markdown ---
#@markdown **Salvar o zip no Google Drive** — e a pasta de destino (criada se não existir).
SAVE_ZIP_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/spectro_calendars"  #@param {type:"string"}

#@markdown **Baixar o zip** — envia o arquivo para o seu computador pelo navegador. Para zips grandes,
#@markdown salvar no Drive e baixar de lá é mais confiável.
DOWNLOAD_ZIP = True  #@param {type:"boolean"}

import html as html_module
import os
import re
import shutil
import zipfile
from pathlib import Path

_audio_element = re.compile(r"<audio\b.*?</audio>", re.S)
_audio_source = re.compile(r'(<source src=")([^"]+)(" type="audio/wav">)')
BUNDLED_AUDIO = {}


def prepare_page(text):
    """Reescreve os links de áudio para a cópia offline: apontando para as gravações
    incluídas no zip, ou removendo os players por completo."""
    if not INCLUDE_AUDIO:
        return text
    if not BUNDLE_AUDIO_IN_ZIP:
        return _audio_element.sub("", text)

    def rewrite(match):
        wav = Path(os.path.normpath(OUT_DIR / html_module.unescape(match.group(2))))
        inside_zip = "audio/" + Path(os.path.relpath(wav, REC_DIR)).as_posix()
        BUNDLED_AUDIO[wav] = inside_zip
        return match.group(1) + html_module.escape(inside_zip, quote=True) + match.group(3)

    return _audio_source.sub(rewrite, text)


ZIP_PATH = Path("/content") / f"{CALENDAR_NAME}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for _file in sorted(OUT_DIR.rglob("*")):
        if not _file.is_file():
            continue
        _arcname = f"{CALENDAR_NAME}/{_file.relative_to(OUT_DIR).as_posix()}"
        if _file.parent == OUT_DIR and _file.name.startswith("index_") and _file.suffix == ".html":
            archive.writestr(_arcname, prepare_page(_file.read_text()))
        else:
            archive.write(_file, _arcname)
    # WAV audio barely compresses, so store it as-is rather than spend the time.
    for _wav, _inside_zip in BUNDLED_AUDIO.items():
        if _wav.exists():
            archive.write(_wav, f"{CALENDAR_NAME}/{_inside_zip}", compress_type=zipfile.ZIP_STORED)
        else:
            print(f"Gravação não encontrada, ignorada: {_wav}")

_size = ZIP_PATH.stat().st_size
print(f"Arquivo zip    : {ZIP_PATH}  ({_size / 2 ** 20:.1f} MB)")
if BUNDLED_AUDIO:
    print(f"Gravações incluídas no zip : {len(BUNDLED_AUDIO)}")
elif INCLUDE_AUDIO:
    print("Os players de áudio foram removidos da página do zip (ative 'incluir o áudio no zip' para mantê-los).")

if SAVE_ZIP_TO_DRIVE:
    _drive_dir = Path(DRIVE_OUTPUT_DIR.strip().rstrip("/"))
    _drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, _drive_dir / ZIP_PATH.name)
    print(f"Salvo no Drive : {_drive_dir / ZIP_PATH.name}")

print("Descompacte e abra o arquivo index_*.html em um navegador.")

if DOWNLOAD_ZIP:
    if _size > 2 * 2 ** 30:
        print("O zip tem mais de 2 GB -- baixe a partir do seu Google Drive, e não pelo navegador.")
    else:
        from google.colab import files

        files.download(str(ZIP_PATH))

---
## Dicas e solução de problemas

**A execução está lenta.** Quase todo o tempo é gasto lendo os arquivos WAV do Google Drive, e não calculando os espectrogramas. Os filtros do Passo 2 são a solução: um passo de tempo de 30 ou 60 minutos, uma janela de horários mais estreita (só o coro do amanhecer, por exemplo) ou um intervalo de datas menor reduzem a transferência na mesma proporção. Manter **usar ffmpeg** ativado também ajuda, e aumentar o número de processos só compensa até o número de CPUs do ambiente (geralmente 2).

**Duas faixas de frequência no mesmo calendário.** Execute os Passos 4–6 duas vezes com o mesmo *nome do calendário*, mas com *rótulos do espectrograma* diferentes (ex.: `lf` com a faixa de 0–2000 Hz e depois `hf` com 2000–20000 Hz). Você terá `index_lf.html` e `index_hf.html` lado a lado, e o zip leva os dois.

**A sessão expira em execuções longas.** O Colab desconecta notebooks ociosos. Execute o Passo 4 novamente com **começar do zero** desmarcado e com o backend ffmpeg: as miniaturas existentes são puladas, então cada execução avança um pouco mais. Manter a aba do navegador aberta e visível ajuda.

**Anotações depois do download.** A exportação em `.txt` também funciona na cópia baixada. Uma ressalva: o Chrome não permite armazenamento local em páginas abertas direto do disco (`file://`), então lá as caixas marcadas duram apenas enquanto a aba estiver aberta — exporte antes de fechar. O Firefox mantém a seleção, assim como a visualização no notebook (Passo 5).

**`No WAV files found`.** Confira o caminho da pasta e ative **incluir subpastas** se as gravações estiverem um nível abaixo. Apenas `.wav`/`.WAV` é lido — `.flac` e `.mp3` são ignorados.

**Um nome de arquivo não pôde ser lido.** O prefixo e o formato de data/hora precisam descrever o nome exatamente: `SM4_20250615_203000.wav` exige o prefixo `SM4_` (com o underline) e o formato `%Y%m%d_%H%M%S`. O Passo 3 mostra um exemplo montado com as suas configurações — compare com um nome de arquivo real.

**Nada aparece no Passo 5.** A visualização precisa que os cookies de terceiros estejam habilitados para `googleusercontent.com`; se o quadro ficar em branco, habilite-os e execute a célula de novo. **Um player diz que não foi possível carregar a gravação:** a mensagem traz o motivo; `HTTP 404` significa que a gravação não está mais no caminho que tinha no Passo 4 (execute os Passos 4 e 5 de novo).